<a href="https://colab.research.google.com/github/2401463-sch/SBE3136-Lab-Project-2401463/blob/main/Project_Car_Dealer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Upload your CSV file
from google.colab import files
uploaded = files.upload()

# Read the CSV (replace filename if needed)
df = pd.read_csv(list(uploaded.keys())[0])

# Show first few rows
df.head()

Saving Dataset for Car - Cleaned Dataset .csv to Dataset for Car - Cleaned Dataset  (2).csv


,car_brand,car_make,car_age,ownership,price,total_coe_left_years,coe,dereg_value,road_tax_per_year,omv,arf,total_mileage_km,transmission,engine_cap_cc,power_kw
0,Mazda,Mazda 3 HB 2.0A Sport Sunroof,7,3,58800,3.49,57501,34170.0,1210,21238,21734,104520.0,Auto,1998,121
1,Mazda,Mazda 3 HB 1.5A Deluxe Sunroof,5,1,85800,5.77,25501,28104.0,682,17872,17872,43272.0,Auto,1496,88
2,Mazda,Mazda 3 HB 1.5A Deluxe Sunroof,5,1,79500,5.43,32311,31145.0,682,18342,18342,66400.0,Auto,1496,88
3,Mazda,Mazda 3 HB 1.5A Sunroof,6,1,56800,4.55,25902,20425.0,182,17329,12329,28633.0,Auto,1496,88
4,Mazda,Mazda 3 HB 1.5A Deluxe Sunroof,8,2,42800,2.63,57301,22920.0,682,18108,13108,89000.0,Auto,1496,88


In [2]:
# Convert everything to lowercase (important for consistency)
df['car_make'] = df['car_make'].str.lower()

# Function to standardise model names
def simplify_model(name):
    if 'mazda 3' in name:
        return 'mazda 3'
    elif 'honda vezel' in name:
        return 'honda vezel'
    elif 'toyota corolla' in name:
        return 'toyota corolla altis'
    else:
        return name

# Apply function
df['model_simplified'] = df['car_make'].apply(simplify_model)

# Check results
df[['car_make', 'model_simplified']].head()

,car_make,model_simplified
0,mazda 3 hb 2.0a sport sunroof,mazda 3
1,mazda 3 hb 1.5a deluxe sunroof,mazda 3
2,mazda 3 hb 1.5a deluxe sunroof,mazda 3
3,mazda 3 hb 1.5a sunroof,mazda 3
4,mazda 3 hb 1.5a deluxe sunroof,mazda 3


In [3]:
!pip install landingai

In [4]:
from landingai.predict import Predictor

# Replace with your actual values
endpoint_id = "3ad32ebe-da0e-4ddf-a7f9-ec379fded40e"
api_key = "land_sk_gcSn8WaIHYgNYTkH2AczetyYcR5nsPNd04ATJHgyNBQDlIDIo1"

predictor = Predictor(endpoint_id, api_key=api_key)

In [20]:
from PIL import Image

# Upload image
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# Open image
image = Image.open(image_path)

# Predict
result = predictor.predict(image)

# Extract label
predicted_label = result[0].label_name.lower()
print("Raw:", predicted_label)

# Simplify
predicted_label = simplify_model(predicted_label)
print("Simplified:", predicted_label)

# Filter dataset
filtered_df = df[df['model_simplified'] == predicted_label]

# Show result
filtered_df.head()

Saving Slide17.JPG to Slide17.JPG
Raw: mazda 3
Simplified: mazda 3


,car_brand,car_make,car_age,ownership,price,total_coe_left_years,coe,dereg_value,road_tax_per_year,omv,arf,total_mileage_km,transmission,engine_cap_cc,power_kw,model_simplified
0,Mazda,mazda 3 hb 2.0a sport sunroof,7,3,58800,3.49,57501,34170.0,1210,21238,21734,104520.0,Auto,1998,121,mazda 3
1,Mazda,mazda 3 hb 1.5a deluxe sunroof,5,1,85800,5.77,25501,28104.0,682,17872,17872,43272.0,Auto,1496,88,mazda 3
2,Mazda,mazda 3 hb 1.5a deluxe sunroof,5,1,79500,5.43,32311,31145.0,682,18342,18342,66400.0,Auto,1496,88,mazda 3
3,Mazda,mazda 3 hb 1.5a sunroof,6,1,56800,4.55,25902,20425.0,182,17329,12329,28633.0,Auto,1496,88,mazda 3
4,Mazda,mazda 3 hb 1.5a deluxe sunroof,8,2,42800,2.63,57301,22920.0,682,18108,13108,89000.0,Auto,1496,88,mazda 3


In [46]:
# Make sure filtered_df is used
data = filtered_df.copy()
data = data.dropna()

# Separate features + target
features = [
    "car_age",
    "ownership",
    "total_mileage_km",
    "engine_cap_cc",
    "power_kw",
    "transmission",
    "coe",
    "total_coe_left_years",
    "dereg_value"
]

X = data[features]
y = data['price']

# Encode
X = pd.get_dummies(X, drop_first=True)

# Train model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

# NOW predict using THIS filtered dataset
sample = X.iloc[0:1]
predicted_price = model.predict(sample)

print("Predicted Price:", predicted_price[0])

Predicted Price: 53350.94843887634


In [63]:
# Example user input (you can change values)
user_input = {
    "car_age": 10,
    "ownership": 1,
    "coe": 40000,
    "dereg_value": 30000,
    "total_mileage_km": 60000,
    "engine_cap_cc": 1500,
    "power_kw": 85,
    "transmission": "auto"
}

In [64]:
user_df = pd.DataFrame([user_input])

In [65]:
print(user_df)
print("Columns:", user_df.columns)

   car_age  ownership    coe  dereg_value  total_mileage_km  engine_cap_cc  \
0       10          1  40000        30000             60000           1500   

   power_kw transmission  
0        85         auto  
Columns: Index(['car_age', 'ownership', 'coe', 'dereg_value', 'total_mileage_km',
       'engine_cap_cc', 'power_kw', 'transmission'],
      dtype='object')


In [66]:
user_df = pd.get_dummies(user_df)

# VERY IMPORTANT
print("Before align:", user_df.shape)

user_df = user_df.reindex(columns=X.columns, fill_value=0)

print("After align:", user_df.shape)

Before align: (1, 8)
After align: (1, 8)


In [67]:
predicted_price = model.predict(user_df)

print("Predicted Price:", predicted_price[0])

Predicted Price: 25336.72436239865


In [68]:
!pip install gradio

  Attempting uninstall: aiofiles
    Found existing installation: aiofiles 0.8.0
    Uninstalling aiofiles-0.8.0:
      Successfully uninstalled aiofiles-0.8.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
landingai 0.3.49 requires aiofiles<1.0.0,>=0.7.0, but you have aiofiles 24.1.0 which is incompatible.


In [73]:
import gradio as gr

def predict_price(image, car_age, mileage, coe, engine_cap, power, dereg):

    # Convert image
    from PIL import Image
    image = Image.fromarray(image)

    # LandingLens prediction
    result = predictor.predict(image)
    label = result[0].label_name.lower()
    label = simplify_model(label)

    # Filter dataset
    filtered_df = df[df['model_simplified'] == label].dropna()

    # Features
    features = [
        "car_age", "ownership", "total_mileage_km",
        "engine_cap_cc", "power_kw", "transmission",
        "coe", "dereg_value"
    ]

    X = filtered_df[features]
    y = filtered_df['price']

    X = pd.get_dummies(X, drop_first=True)

    from sklearn.linear_model import LinearRegression
    model = LinearRegression()
    model.fit(X, y)

    # User input
    user_input = {
        "car_age": car_age,
        "ownership": 1,
        "total_mileage_km": mileage,
        "engine_cap_cc": engine_cap,
        "power_kw": power,
        "transmission": "auto",
        "coe": coe,
        "dereg_value": dereg
    }

    user_df = pd.DataFrame([user_input])
    user_df = pd.get_dummies(user_df)
    user_df = user_df.reindex(columns=X.columns, fill_value=0)

    price = model.predict(user_df)[0]

    return f"Model: {label} | Predicted Price: ${price:.2f}"

In [74]:
interface = gr.Interface(
    fn=predict_price,
    inputs=[
        gr.Image(type="numpy"),
        gr.Number(label="Car Age"),
        gr.Number(label="Mileage"),
        gr.Number(label="COE"),
        gr.Number(label="Engine CC"),
        gr.Number(label="Power (kW)"),
        gr.Number(label="Dereg Value")
    ],
    outputs="text",
    title="Car Price Predictor"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://52b4b71e3e25778f33.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
